In [1]:
# CELL 1: Setup môi trường & nạp module RAG Core
import os
import sys
import json
import asyncio

# Trỏ về root của dev_llm_service
sys.path.append(os.path.abspath(".."))

from app.core.config import settings
from app.core.database import engine, SessionLocal
from app.routers.dependencies import get_embedding_service, get_vector_retriever, get_chat_model
from sqlalchemy import text

print("=" * 65)
print("✅ Môi trường Python & RAG Core đã nạp thành công!")
print(f"📌 Vector Database    : SQL Server")
print(f"📌 Embedding Model    : {getattr(settings, 'EMBEDDING_MODEL', 'BAAI/bge-m3')}")
print(f"📌 TEI URL            : {settings.TEI_URL}")
print(f"📌 Reranker Type      : {settings.RERANKER_TYPE}")
print("=" * 65)

c:\2_Company\GSOFT\Enterprice-Chatbot\BVBank-Chatbot\dev_llm_service\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Môi trường Python & RAG Core đã nạp thành công!
📌 Vector Database    : SQL Server
📌 Embedding Model    : bge-m3:latest
📌 TEI URL            : http://localhost:8080
📌 Reranker Type      : disabled


In [2]:
# CELL 2: Quét các chunks chứa rác nhị phân (bắt đầu bằng 'PK\x03\x04')
with engine.connect() as conn:
    # Tìm các chunks có chứa header ZIP PK\x03\x04
    query = text("""
        SELECT 
            JSON_VALUE(metadata, '$.backendId') as doc_id,
            JSON_VALUE(metadata, '$.source') as file_name,
            COUNT(*) as garbage_chunk_count
        FROM Documents
        WHERE document LIKE 'PK%' OR document LIKE '%word/_rels%'
        GROUP BY JSON_VALUE(metadata, '$.backendId'), JSON_VALUE(metadata, '$.source')
    """)
    garbage_docs = conn.execute(query).fetchall()

print("🔍 KẾT QUẢ QUÉT TÀI LIỆU CHỨA RÁC NHỊ PHÂN TRONG CSDL:")
print("-" * 65)
if not garbage_docs:
    print("🎉 Tuyệt vời! Không tìm thấy tài liệu nào bị lỗi nhị phân PK.")
else:
    for row in garbage_docs:
        print(f"❌ Doc ID: {row[0]} | File: {row[1]} | Số chunks rác: {row[2]}")

print("\n👉 Xem thử nội dung 2 chunk rác đầu tiên của Doc 1055:")
with engine.connect() as conn:
    samples = conn.execute(text("SELECT TOP 2 id, document FROM Documents WHERE id LIKE '1055_%'")).fetchall()
    for s in samples:
        print(f"\n[Chunk ID: {s[0]}] Độ dài: {len(s[1])} ký tự")
        print(f"Preview (50 ký tự đầu): {repr(s[1][:50])}")

🔍 KẾT QUẢ QUÉT TÀI LIỆU CHỨA RÁC NHỊ PHÂN TRONG CSDL:
-----------------------------------------------------------------
❌ Doc ID: 1055 | File: HDSD Phan mem QLTS GD 1-2-3 - DANH MUC - HE THONG.docx | Số chunks rác: 7
❌ Doc ID: 1057 | File: HDSD Phan mem QLTS GD 1-2-3 - DANH MUC - HE THONG.docx | Số chunks rác: 7

👉 Xem thử nội dung 2 chunk rác đầu tiên của Doc 1055:

[Chunk ID: 1055_0] Độ dài: 600 ký tự
Preview (50 ký tự đầu): 'PK\x03\x04\x14\x00\x06\x00\x08\x00V,V\x01\x00\x00\t\x00\x00\x13\x00\x08\x02[Content_Types].xml \x04\x02(\x00\x02\x00\x00'

[Chunk ID: 1055_1] Độ dài: 600 ký tự
Preview (50 ký tự đầu): '\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'


In [3]:
# CELL 3: So sánh Naive Chunking (cũ) và Semantic Slide-Level (mới)
from app.ai.rag.chunking.splitter import TextChunker
from pptx import Presentation

# Lấy 1 file PPTX thực tế trong kho my_documents
sample_pptx = os.path.abspath(os.path.join("..", "my_documents", "Slide dao tao Phan mem QLTS GD 2 - CAP PHE DUYET - KE HOACH - MUA SAM - THANH QUYET TOAN.pptx"))

prs = Presentation(sample_pptx)
print(f"📁 Khảo sát file: {os.path.basename(sample_pptx)} ({len(prs.slides)} slides)\n")

# 1. Trích xuất thô kiểu cũ
all_slide_text = []
for s in prs.slides:
    for shape in s.shapes:
        if shape.has_text_frame:
            for p in shape.text_frame.paragraphs:
                if p.text.strip():
                    all_slide_text.append(p.text.strip())

raw_full_text = "\n".join(all_slide_text)

# Chạy TextChunker cũ (600 chars, overlap 120)
chunker = TextChunker(chunk_size=600, chunk_overlap=120)
naive_chunks = chunker.split_text(raw_full_text)

print("🔴 [CƠ CHẾ CŨ]: Ghép 99 slide thành 1 file dài -> Cắt 600 ký tự:")
print(f"   Tổng số chunks tạo ra: {len(naive_chunks)}")
print("   Ví dụ 1 chunk bị cắt cụt từ ngữ:")
for c in naive_chunks:
    if c.startswith("ớc") or c.startswith("ện") or c.endswith("Tro") or c.endswith("chợ"):
        print("-" * 50)
        print(f"Đoạn đầu: '{c.split()[0]}' | Đoạn cuối: '{c.split()[-1]}'")
        print(c[:200] + " ... " + c[-100:])
        print("-" * 50)
        break

# 2. Cơ chế mới: 1 Slide = 1 Vector trọn vẹn (Atomic Semantic Unit)
print("\n🟢 [CƠ CHẾ MỚI]: 1 Slide = 1 Chunk Độc Lập + Header Ngữ Cảnh:")
slide_20 = prs.slides[19] # Slide số 20
slide_20_texts = []
for shape in slide_20.shapes:
    if shape.has_text_frame:
        for p in shape.text_frame.paragraphs:
            if p.text.strip():
                slide_20_texts.append(p.text.strip())

enriched_slide_20 = (
    f"[Tài liệu: {os.path.basename(sample_pptx)}] [Slide 20/{len(prs.slides)}]\n"
    f"### Tiêu đề: {slide_20.shapes.title.text.strip() if slide_20.shapes.title else 'N/A'}\n"
    + "\n".join(slide_20_texts)
)
print(enriched_slide_20)
print(f"\n=> Độ dài: {len(enriched_slide_20)} ký tự (Không bị cắt xén, không đứt bước 1, 2, 3!)")

📁 Khảo sát file: Slide dao tao Phan mem QLTS GD 2 - CAP PHE DUYET - KE HOACH - MUA SAM - THANH QUYET TOAN.pptx (99 slides)

🔴 [CƠ CHẾ CŨ]: Ghép 99 slide thành 1 file dài -> Cắt 600 ký tự:
   Tổng số chunks tạo ra: 69
   Ví dụ 1 chunk bị cắt cụt từ ngữ:

🟢 [CƠ CHẾ MỚI]: 1 Slide = 1 Chunk Độc Lập + Header Ngữ Cảnh:
[Tài liệu: Slide dao tao Phan mem QLTS GD 2 - CAP PHE DUYET - KE HOACH - MUA SAM - THANH QUYET TOAN.pptx] [Slide 20/99]
### Tiêu đề: 1. PHÂN HỆ QUẢN LÝ KẾ HOẠCH
1. PHÂN HỆ QUẢN LÝ KẾ HOẠCH
Kiểm soát  viên điều phối tờ trình - Quản lý kế hoạch\Điều phối công việc
Bước 1: Đăng nhập hệ thống
Bước 2: Chọn mục Quản lý kế hoạch/Điều phối công việc trong màn hình Tìm kiếm thông tin, có thể nhập các thông tin tìm kiếm như mã số tờ trình, tên tờ trình v.v.. và click nút Tìm kiếm                     để tìm.
Bước 4: Chọn tờ trình cần điều chuyển cho nhân viên xử lý bằng cách tích vào           ở đầu  hàng trên lưới: chọn Người được giao xử lý và vai trò
Bước 5: Click nút save            

In [4]:
# CELL 4: Test Retrieval xem các đoạn văn bản được bốc lên
retriever = get_vector_retriever()
test_query = "Quy trình phê duyệt tờ trình mua sắm"

print(f"🔍 Đang truy vấn CSDL cho câu hỏi: '{test_query}'...\n")

search_results = await retriever.retrieve_context(
    query=test_query,
    top_k=5,
    user_roles="Admin,Staff",
    user_department="Kế toán"
)

docs = search_results.get("documents", [[]])[0]
citations = search_results.get("citations", [[]])[0]

print(f"✅ Tìm thấy {len(docs)} đoạn tài liệu được xếp hạng cao nhất:")
for idx, (doc, cit) in enumerate(zip(docs, citations), start=1):
    source = cit.get("source") or cit.get("document_name")
    page = cit.get("page")
    print(f"\n[{idx}] Nguồn: {source} (Trang/Slide: {page}) | Độ dài chunk: {len(doc)} chars")
    print("┌" + "─" * 60)
    print(doc[:300] + ("..." if len(doc) > 300 else ""))
    print("└" + "─" * 60)

🔍 Đang truy vấn CSDL cho câu hỏi: 'Quy trình phê duyệt tờ trình mua sắm'...

✅ Tìm thấy 5 đoạn tài liệu được xếp hạng cao nhất:

[1] Nguồn: Slide dao tao Phan mem QLTS GD 3 - APP KIEM KE & PHE DUYET.pptx (Trang/Slide: 26) | Độ dài chunk: 600 chars
┌────────────────────────────────────────────────────────────
 đã được tạo trên web.
1. CHỨC NĂNG PHÊ DUYỆT
DUYỆT PHIẾU HỢP ĐỒNG MUA SẮM
1. CHỨC NĂNG PHÊ DUYỆT
Các bước để xét duyệt một tờ trình chủ trương:
Điều kiện: Đăng nhập thành công vào hệ thống và chọn “Chức năng phê duyệt”.
Bước 3: Hiện màn hình chi tiết hợp đồng mua sắm, ấn phê duyệt để tiến hành phê ...
└────────────────────────────────────────────────────────────

[2] Nguồn: Slide dao tao Phan mem QLTS GD 2 - CAP PHE DUYET - KE HOACH - MUA SAM - THANH QUYET TOAN.pptx (Trang/Slide: 36) | Độ dài chunk: 600 chars
┌────────────────────────────────────────────────────────────
i với trường hợp PYC mua sắm có có hình thức là chỉ định thầu
2. PHÂN HỆ QUẢN LÝ MUA SẮM
Chức năng phê duyệt phi

In [5]:
# CELL 5: Test Cross-Encoder Reranker
from app.ai.rag.reranker import get_reranker

reranker = get_reranker(reranker_type="bge", model_name="BAAI/bge-reranker-base")

if docs:
    print(f"🤖 Đang chấm điểm lại {len(docs)} chunks bằng Cross-Encoder...")
    ranked_pairs = reranker.rerank_with_scores(
        query=test_query,
        chunks=docs,
        top_k=len(docs)
    )
    
    print("\n📊 BẢNG SO SÁNH THỨ TỰ TRƯỚC VÀ SAU KHI RERANK:")
    print(f"{'Vị trí Mới':<12} | {'Vị trí Cũ (RRF)':<16} | {'Điểm Cross-Encoder':<20} | {'Đoạn trích dẫn'}")
    print("-" * 80)
    for new_rank, (old_idx, score) in enumerate(ranked_pairs, start=1):
        snippet = docs[old_idx][:50].replace("\n", " ")
        print(f"Top #{new_rank:<7} | Top #{old_idx + 1:<11} | Score: {score:<13.4f} | {snippet}...")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5209.99it/s]


🤖 Đang chấm điểm lại 5 chunks bằng Cross-Encoder...

📊 BẢNG SO SÁNH THỨ TỰ TRƯỚC VÀ SAU KHI RERANK:
Vị trí Mới   | Vị trí Cũ (RRF)  | Điểm Cross-Encoder   | Đoạn trích dẫn
--------------------------------------------------------------------------------
Top #1       | Top #1           | Score: 0.9891        |  đã được tạo trên web. 1. CHỨC NĂNG PHÊ DUYỆT DUYỆ...
Top #2       | Top #4           | Score: 0.9858        | hân quyền chức năng duyệt Kết quả mong muốn: người...
Top #3       | Top #5           | Score: 0.9718        | DUYỆT TỜ TRÌNH CHỦ TRƯƠNG Mục đích: cho phép người...
Top #4       | Top #2           | Score: 0.9523        | i với trường hợp PYC mua sắm có có hình thức là ch...
Top #5       | Top #3           | Score: 0.9319        | ung tờ trình hoặc trong năm, trình trạng duyệt, đơ...


In [6]:
# CELL 6: Test Generation từ Context sạch
from app.ai.rag.context.builder import RagContextBuilder

# 1. Đóng gói context với trích dẫn nguồn
context_text, formatted_citations = RagContextBuilder.build_context(
    documents=docs[:3],
    citations=citations[:3],
    max_tokens=3000
)

print("📝 CONTEXT ĐƯỢC ĐƯA VÀO PROMPT CHO LLM:")
print("=" * 65)
print(context_text[:800] + "\n...(còn tiếp)...")
print("=" * 65)

# 2. Gọi LLM sinh câu trả lời
llm = get_chat_model()
system_msg = (
    "Bạn là trợ lý AI chuyên gia quy chế và quy trình ngân hàng BVBank & gAMSPro. "
    "Hãy trả lời câu hỏi của người dùng CHỈ dựa trên ngữ cảnh tài liệu được cung cấp dưới đây. "
    "Trình bày rõ ràng theo từng bước 1, 2, 3."
)
user_msg = f"Ngữ cảnh tài liệu:\n{context_text}\n\nCâu hỏi: {test_query}"

print("\n⏳ LLM đang suy luận và sinh câu trả lời...")
response = await llm.ainvoke([("system", system_msg), ("user", user_msg)])

print("\n🎯 KẾT QUẢ TRẢ LỜI CỦA RAG AGENT:")
print("-" * 65)
print(response.content)
print("-" * 65)

AttributeError: type object 'RagContextBuilder' has no attribute 'build_context'